# Lab 01 · Linear and logistic regression

**Audience:** master's students in Computer Science. **Time:** about 120 minutes.

- **Example A (35 minutes):** predict request latency with a simple linear model.
- **Example B (75 minutes):** predict latency-budget violations and choose an alert threshold.
- **Review (10 minutes):** explain errors and check the evaluation protocol.

The examples use original synthetic service data with fixed seeds. They are controlled
teaching examples, not measurements from a production service. No downloads are needed
after the course dependencies are installed.

**Deliverables:** complete tasks A1–A4 and B1–B5, run the notebook from a fresh kernel,
and fill in the written answers. Submit the executed notebook. Consult the accompanying
`lab_linear_logistic_regression.pdf` for the brief and assessment criteria.

## Student version

Code cells tagged **exercise** deliberately contain `NotImplementedError`. Replace the
TODO block with your implementation, using the exact variable names specified. Start
with A1 and work in order; later cells depend on earlier tasks. Supplied checks describe
expected shapes and behavior, not the missing implementation. Do not change seeds,
splits, the threshold grid, or test data to improve a reported result.

Run supplied setup and data cells as they are. They create the examples and plotting
helpers so you can focus on modeling decisions. This unfinished notebook is expected
to stop at A1 until you implement it.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display
from scipy.special import expit
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (mean_absolute_error, mean_squared_error, accuracy_score,
    precision_score, recall_score, confusion_matrix, log_loss,
    ConfusionMatrixDisplay)

SEED = 42
plt.rcParams.update({"figure.figsize": (8, 4.5), "figure.dpi": 110,
    "axes.grid": True, "grid.alpha": 0.2, "axes.spines.top": False,
    "axes.spines.right": False, "font.size": 11})
print(f"Python {sys.version.split()[0]} | NumPy {np.__version__} | scikit-learn {sklearn.__version__}")

## Example A · How long will a request take?

A service processes requests of different sizes. Build a model whose input is
`size_mb` and whose target is `latency_ms`. The hidden average in this simulation is
linear, but individual requests vary. In a real system, the mean relationship is not
known in advance.

**Model:** predicted latency = intercept + slope × request size.

An intercept has units of milliseconds; a slope has units of milliseconds per MB.
The loss used for fitting is squared error. We will report MAE and RMSE on unseen
requests and compare with a model that always predicts the training mean.

In [ ]:
# Supplied synthetic data. No model has been fitted yet.
rng_a = np.random.default_rng(7)
requests = pd.DataFrame({"size_mb": rng_a.uniform(0.5, 20, 120)})
requests["latency_ms"] = 25 + 5*requests["size_mb"] + rng_a.normal(0, 8, len(requests))
display(requests.head())
print("Observations:", len(requests))

### A1

Create `Xa` as a DataFrame containing only `size_mb`, and `ya` as the `latency_ms` Series.
Split into `Xa_train`, `Xa_test`, `ya_train`, `ya_test` using `test_size=0.25` and
`random_state=SEED`. Keep the original row indices. Make a scatter plot of **training**
size against training latency, with axis labels and units. Describe the pattern before fitting.

In [ ]:
# TODO A1: select inputs and target, split, and plot training data.
# Hint: use double brackets for a one-column feature DataFrame.
raise NotImplementedError("Complete A1: data selection, split, and scatter plot")

In [ ]:
# Supplied checks: keep the target out of the input features and the split disjoint.
assert list(Xa.columns) == ["size_mb"]
assert len(Xa_train) == 90 and len(Xa_test) == 30
assert set(Xa_train.index).isdisjoint(Xa_test.index)
assert Xa_train.index.equals(ya_train.index) and Xa_test.index.equals(ya_test.index)

### A2

Fit `linear_model = LinearRegression()` and `mean_model = DummyRegressor(strategy="mean")`
on training data. Create `pred_a` and `baseline_a`, their predictions on `Xa_test`.
Display the learned intercept and slope. Do not fit on `Xa_test` or use `ya_test.mean()`
for the baseline.

In [ ]:
# TODO A2: fit both models on training rows and predict the test rows.
raise NotImplementedError("Complete A2: linear model and baseline")

In [ ]:
assert np.asarray(pred_a).shape == (30,)
assert np.isfinite(pred_a).all()
assert np.allclose(baseline_a, ya_train.mean()), "Baseline must use the training mean."

### A3

Create `regression_results`, a DataFrame indexed by `Baseline` and `Linear regression`,
with columns `MAE_ms` and `RMSE_ms`. Compute both metrics on the same test rows.
RMSE is `np.sqrt(mean_squared_error(...))`. Plot training points, test points, the fitted
line, and the baseline over the observed size range. Label the axes and legend.

In [ ]:
# TODO A3: compare held-out errors and draw both prediction rules.
raise NotImplementedError("Complete A3: evaluation table and fitted-line figure")

**A3 discussion — explain your result.** Explain the intercept and slope with units. Does request size help on the test set?
Why are individual points allowed to miss the fitted line?

**Your answer:**

_Write 2–4 sentences here._

### A4

Create `residual_a = ya_test - pred_a`. Draw residuals against test request size and
include a horizontal zero line. Create `latency_4mb`, the predicted latency for a 4 MB
request, using a DataFrame with the same feature name as the training data.

In [ ]:
# TODO A4: inspect residuals and make a prediction for a new request.
raise NotImplementedError("Complete A4: residual plot and 4 MB prediction")

**A4 discussion — explain your result.** What does a positive residual mean? Does the residual plot establish that every model
assumption is correct? Would you trust this line for a 1,000 MB request?

**Your answer:**

_Write 2–4 sentences here._

## Example B · Which requests should trigger a latency alert?

Before processing a request, predict whether it will exceed a latency budget. Each row
contains four features available at arrival:

| Feature | Meaning |
|---|---|
| `payload_mb` | Request payload in MB |
| `queue_length` | Number of requests already waiting |
| `cpu_percent` | CPU utilization at arrival, from 0 to 100 |
| `cache_hit` | Whether a cache lookup already found a reusable result (0/1) |

The target `budget_violation` is 1 for a slow request and 0 otherwise. These are synthetic
Bernoulli outcomes: even with all four inputs, the result has uncertainty. The generator
is supplied for transparency; do not use its internal score or probabilities as extra
features or as your model's predictions.

**Operational goal:** an alert is a positive decision. A false alert costs **1 unit**;
a missed violation costs **5 units**. Correct decisions cost zero. Compare total cost
`FP + 5 × FN` as well as accuracy, precision, and recall. A threshold of 0.5 is a
reference policy, not an automatically optimal choice.

In [ ]:
# Supplied simulator. Four observed features; one stochastic outcome.
rng_b = np.random.default_rng(2026)
n_requests = 1200
traffic = pd.DataFrame({
    "payload_mb": np.clip(rng_b.lognormal(mean=2.0, sigma=0.7, size=n_requests), 0.5, 80),
    "queue_length": np.clip(rng_b.gamma(shape=2, scale=8, size=n_requests).astype(int), 0, 80),
    "cpu_percent": rng_b.uniform(10, 95, n_requests),
    "cache_hit": rng_b.binomial(1, 0.65, n_requests),
})
_simulation_score = (-4.8 + 0.08*traffic.payload_mb + 0.065*traffic.queue_length
                     + 0.04*traffic.cpu_percent - 1.0*traffic.cache_hit)
traffic["budget_violation"] = rng_b.binomial(1, expit(_simulation_score))
del _simulation_score  # The generator's hidden rule is not an input to the learning task.
FEATURES = ["payload_mb", "queue_length", "cpu_percent", "cache_hit"]
FP_COST, FN_COST = 1, 5
THRESHOLDS = np.round(np.arange(0.05, 1.0, 0.05), 2)
display(traffic.head())

### B1

Select `Xb = traffic[FEATURES]` and `yb = traffic["budget_violation"]`.
First reserve 20% as `Xb_test`, `yb_test`, leaving `Xb_dev`, `yb_dev`. Then reserve 25%
of development rows as `Xb_val`, `yb_val`, leaving `Xb_train`, `yb_train`.
Use `random_state=SEED` and stratify on the corresponding target in **both** splits.
This gives a 60% / 20% / 20% train/validation/test split. Keep row indices intact.
Display the target counts and feature scales for **training data** only.

In [ ]:
# TODO B1: two stratified splits; inspect training target balance and feature scales.
raise NotImplementedError("Complete B1: train/validation/test separation")

In [ ]:
assert [len(Xb_train), len(Xb_val), len(Xb_test)] == [720, 240, 240]
assert list(Xb_train.columns) == FEATURES
for left, right in [(Xb_train, Xb_val), (Xb_train, Xb_test), (Xb_val, Xb_test)]:
    assert set(left.index).isdisjoint(right.index), "Splits must not share rows."
assert set(Xb_train.index) | set(Xb_val.index) | set(Xb_test.index) == set(traffic.index)
for inputs, target in [(Xb_train,yb_train), (Xb_val,yb_val), (Xb_test,yb_test)]:
    assert inputs.index.equals(target.index)

**B1 discussion — explain your result.** Why is an always-no-alert baseline potentially accurate but operationally expensive?
Why does a random split suit this simulator but not necessarily real service logs?

**Your answer:**

_Write 2–4 sentences here._

### B2

Build `classifier_pipeline = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))`.
Create `classifier_search = GridSearchCV(...)` with `logisticregression__C` in
`[0.01, 0.1, 1, 10]`, five-fold `StratifiedKFold(shuffle=True, random_state=SEED)`, and
`scoring="neg_log_loss"`. Fit it on `Xb_train, yb_train` **only**.
Set `selected_model = classifier_search.best_estimator_` and show the chosen C and
mean CV log loss for each candidate. The search's final refit remains within the training partition.

In [ ]:
# TODO B2: search a scaling + logistic pipeline using training-only cross-validation.
raise NotImplementedError("Complete B2: scaled logistic model and CV selection")

In [ ]:
assert np.allclose(selected_model.named_steps["standardscaler"].mean_, Xb_train.mean().to_numpy())
assert selected_model.named_steps["logisticregression"].classes_.tolist() == [0, 1]

**B2 discussion — explain your result.** Why place the scaler inside the pipeline passed to cross-validation? Does a smaller C
mean weaker or stronger regularization? Why choose C with a probability loss here?

**Your answer:**

_Write 2–4 sentences here._

### B3

Create `val_probability = selected_model.predict_proba(Xb_val)[:, 1]` and
`val_labels_05` using threshold 0.5. Build `validation_summary_05`, a Series with
`accuracy`, `precision`, `recall`, and `log_loss`. Use `zero_division=0` for precision
and recall. Draw probability histograms for actual negative and positive validation
requests on common bins, and mark threshold 0.5.

In [ ]:
# TODO B3: evaluate validation probabilities and the reference decision threshold.
raise NotImplementedError("Complete B3: validation probabilities, metrics, and histogram")

In [ ]:
assert val_probability.shape == (240,)
assert np.isfinite(val_probability).all() and ((val_probability >= 0) & (val_probability <= 1)).all()

### B4

For each value in `THRESHOLDS`, convert validation probabilities to labels with `>=`.
Create `threshold_results` with columns `threshold`, `FP`, `FN`, and `cost`, where
`cost = FP_COST*FP + FN_COST*FN`. Use `confusion_matrix(..., labels=[0, 1]).ravel()`
to obtain `tn, fp, fn, tp`. Choose `best_threshold` with the smallest validation cost;
**if costs tie, choose the largest threshold**. Plot validation cost against threshold,
mark the selection, and display FP/FN counts at 0.5 and at the selected threshold.
Do not use test predictions or test labels for this task.

In [ ]:
# TODO B4: evaluate the fixed threshold grid and select by validation cost.
raise NotImplementedError("Complete B4: cost table, selection, and plot")

In [ ]:
assert len(threshold_results) == len(THRESHOLDS)
assert np.allclose(threshold_results["cost"], FP_COST*threshold_results.FP + FN_COST*threshold_results.FN)
minimum_cost = threshold_results["cost"].min()
assert best_threshold == threshold_results.loc[threshold_results.cost == minimum_cost, "threshold"].max()

**B4 discussion — explain your result.** Explain the selected threshold's tradeoff relative to 0.5. Why is a lower validation
cost not a guarantee of the same improvement on test data? Does changing the threshold
change the probabilities?

**Your answer:**

_Write 2–4 sentences here._

### Freeze before the final evaluation

At this point both the model and `best_threshold` are fixed. **Do not refit on train +
validation** in this lab: a refit could change the probability scale and the meaning of
the selected threshold. Evaluate these three predeclared policies on the same test set:

1. Always predict no alert.
2. The fitted logistic model with threshold 0.5.
3. The same fitted logistic model with `best_threshold`.

The two logistic policies have identical probability predictions and log loss. Their
confusion matrices, accuracy, precision, recall, and cost can differ. We do not report
log loss for the always-negative hard-decision baseline: its purpose here is a policy-cost comparison.

### B5

Create `test_probability` once from `selected_model` and `Xb_test`.
Create `test_results`, indexed by `Always no alert`, `Threshold 0.5`, and
`Validation threshold`, with columns `accuracy`, `precision`, `recall`, `FP`, `FN`,
`cost`, and `cost_per_request`. Use the three predeclared policies above.
Print the logistic model's test log loss separately. Plot the two logistic confusion
matrices side by side with labels `No violation` and `Violation`. Do not retune anything
in response to the test comparison.

In [ ]:
# TODO B5: evaluate the frozen policies once on the test set.
raise NotImplementedError("Complete B5: final policy comparison and confusion matrices")

In [ ]:
assert test_probability.shape == (240,)
assert test_results.index.tolist() == ["Always no alert", "Threshold 0.5", "Validation threshold"]
assert np.allclose(test_results.cost_per_request, test_results.cost/len(yb_test))
assert test_results.loc["Always no alert", "recall"] == 0
assert test_results.loc["Always no alert", "cost"] == FN_COST * int(yb_test.sum())
# Test performance is evidence, not a condition to optimize until this notebook passes.

**B5 discussion — explain your result.** Recommend a policy under the stated costs, referring to the validation choice and the
held-out evidence. Is the highest-accuracy policy also the lowest-cost one? Explain
why the two logistic policies share log loss, and name one limitation of this experiment.

**Your answer:**

_Write 2–4 sentences here._

## Optional discussion (not required for completion)

1. If false positives cost five units and false negatives cost one, which direction
   would you expect the threshold to move? Investigate using validation data only.
2. With calibrated probabilities and costs `c_FP` and `c_FN`, compare the expected cost
   of the two decisions and show why the theoretical threshold is
   `c_FP / (c_FP + c_FN)`. Why might a finite validation-grid optimum differ?
3. If the same client submits many correlated requests, how should the splitting
   strategy change? What if CPU behavior changes after a software deployment?

## Submission checklist

- Complete all nine TODO cells and six written-answer prompts.
- Keep the supplied seeds, features, split sizes, threshold grid, and error costs.
- Show labeled figures and the regression, validation, and test comparison tables.
- Restart and run the completed notebook from top to bottom; keep outputs saved.
- Explain errors in milliseconds for A and in decision consequences for B.

**References:** the examples and simulators are original for this lab. API background:
[linear models](https://scikit-learn.org/stable/modules/linear_model.html),
[pipelines](https://scikit-learn.org/stable/modules/compose.html), and
[model evaluation](https://scikit-learn.org/stable/modules/model_evaluation.html).
The companion course notebook supplies the conceptual explanations.